In [47]:
import os

In [96]:
import pandas as pd
import numpy as np
import librosa

In [2]:
import torch, torchaudio
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torchaudio.transforms import MelSpectrogram, AmplitudeToDB
from torch.utils.tensorboard.writer import SummaryWriter
from typing import cast

"""
-----------------------------------------------------------------------------------------------
Step 1: Load dataset SPEECHCOMMANDSDataset [20 pts]
Load dataset from torchaudio.datasets;
Preprocess the dataset to get spectrogram by applying the short-time Fourier transform;
Dynamically generate label mapping from dataset.
-----------------------------------------------------------------------------------------------
"""
# Define the dataset class
class SPEECHCOMMANDSDataset(torch.utils.data.Dataset):
    def __init__(self, dataset: Subset):
        self.dataset = dataset
        self.mel_spectrogram = MelSpectrogram(n_fft=256, hop_length=128)
        self.amplitude_to_db = AmplitudeToDB()

        # Generate label mapping dynamically
        self.label_mapping = self.generate_label_mapping()
        print("Generated label mapping:", self.label_mapping)

    def generate_label_mapping(self):
        """Create a mapping from unique labels to integers."""
        unique_labels = set()
        for _, _, label, *_ in self.dataset:  # Iterate through the dataset
            unique_labels.add(label)
        return {label: idx for idx, label in enumerate(sorted(unique_labels))}

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, idx: int | list[int]) -> tuple[torch.Tensor, int]:
        """ TODO Load and preprocess a single sample from the dataset."""
        waveform, sample_rate, label, *_ = self.dataset[idx]

        # Map label to integer
        if label in self.label_mapping:
            label = self.label_mapping[label]
        else:
            raise ValueError(f"Unexpected label: {label}")

        # Pad or truncate waveform to the same length
        waveform = self.pad_waveform(waveform)

        # Generate spectrogram
        spectrogram = self.amplitude_to_db(self.mel_spectrogram(waveform)) # Convert amplitude to decibels

        # pad spectrogram from (1, 128, 126) to (1, 128, 128)
        spectrogram = torch.nn.functional.pad(spectrogram, (0, 2))
        return spectrogram, label

    @staticmethod
    def pad_waveform(waveform: torch.Tensor, length: int = 16000) -> torch.Tensor:
        """ TODO Pad or truncate waveform to the same length."""
        if waveform.size(-1) < length:
            waveform = torch.nn.functional.pad(waveform, (0, length - waveform.size(-1)))
        else:
            waveform = waveform[:, :length]

        return waveform

In [5]:
# Load the dataset
#Data must be in a genres folder
dataset_path = "./gtzan_300ms_150ms_overlap/data"
dataset = torchaudio.datasets.GTZAN(root=dataset_path, download=False)
total_size = len(dataset)

# Split dataset into train, validation, and test sets
train_size = int(0.8 * total_size)  # Fill in the train size fraction
test_size = total_size - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# Apply the custom preprocessing pipeline
#train_dataset = SPEECHCOMMANDSDataset(train_dataset)
#val_dataset = SPEECHCOMMANDSDataset(val_dataset)
test_dataset = SPEECHCOMMANDSDataset(test_dataset)

#dl_batch_size = 128

# Create DataLoaders
#train_loader = DataLoader(train_dataset, batch_size=dl_batch_size, shuffle=True)
#val_loader = DataLoader(val_dataset, batch_size=dl_batch_size, shuffle=True)
#test_loader = DataLoader(test_dataset, batch_size=dl_batch_size, shuffle=True)

/opt/anaconda3/envs/csc_7760_final/lib/python3.12/site-packages/torchaudio/functional/functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (129) may be set too low.
  warnings.warn(


Generated label mapping: {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}


In [10]:
x,y = test_dataset[0]
x,y

(tensor([[[-100.0000, -100.0000, -100.0000,  ..., -100.0000,    0.0000,
              0.0000],
          [-100.0000, -100.0000, -100.0000,  ..., -100.0000,    0.0000,
              0.0000],
          [-100.0000, -100.0000, -100.0000,  ..., -100.0000,    0.0000,
              0.0000],
          ...,
          [ -36.5762,  -66.2043,  -61.8965,  ..., -100.0000,    0.0000,
              0.0000],
          [ -37.5452,  -75.9343,  -78.7267,  ..., -100.0000,    0.0000,
              0.0000],
          [ -37.7708,  -75.9649,  -84.6385,  ..., -100.0000,    0.0000,
              0.0000]]]),
 9)

In [13]:
# Flatten to a 1D array
flattened = x.view(-1).numpy()

# Create a single-row DataFrame
df = pd.DataFrame([flattened])
df
#df.to_csv("audio_features_dataset.csv", mode='a', header=False, index=False)

,0,1,2,3,4,5,6,7,8,9,...,16374,16375,16376,16377,16378,16379,16380,16381,16382,16383
0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,...,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,-100.0,0.0,0.0


In [3]:
import torchaudio
import torchaudio.transforms as T

audio_file = "gtzan_300ms/data/genres/blues/blues.00000.wav"

# 1. Load the audio file
# waveform is a tensor of shape (channels, time)
waveform, sample_rate = torchaudio.load(audio_file)

# 2. Define the MelSpectrogram transform
mel_spectrogram_transform = T.MelSpectrogram(
    sample_rate=sample_rate,
    n_fft=1024,
    hop_length=512,
    n_mels=128
)

# 3. Perform the transformation
mel_spectrogram = mel_spectrogram_transform(waveform)
mel_spectrogram

tensor([[[1.9526e+01, 3.6210e+00, 2.8339e+00,  ..., 3.2266e+01,
          4.7691e+01, 2.2149e+01],
         [1.6903e+01, 8.1737e+00, 7.4553e+00,  ..., 2.7604e+02,
          3.8346e+02, 4.1563e+01],
         [1.2804e+01, 1.6041e+01, 1.5424e+01,  ..., 6.9401e+02,
          9.5925e+02, 7.5244e+01],
         ...,
         [8.3310e-04, 3.3020e-06, 5.3269e-06,  ..., 2.8904e-05,
          2.3091e-05, 1.8513e-05],
         [8.4557e-04, 8.2368e-07, 4.4219e-07,  ..., 2.2715e-06,
          2.7038e-06, 3.0340e-06],
         [8.3832e-04, 3.2679e-07, 2.3621e-07,  ..., 5.0371e-07,
          4.5703e-07, 1.6808e-06]]])

In [104]:
sec3 = pd.read_csv("Data/features_3_sec.csv")
#Replacing Harmony and Perceptr with Spectral Contrast and Spectral Flatness
sec3.rename(columns={
    'harmony_mean' : 'spectral_contrast_mean',
    'harmony_var'  : 'spectral_contrast_var',
    'perceptr_mean': 'spectral_flatness_mean',
    'perceptr_var' : 'spectral_flatness_var'
}, inplace=True)

sec3.drop('length', axis=1, inplace=True)

features_list = sec3.columns.tolist()
features = sec3.iloc[:0]
features_list

['filename',
 'chroma_stft_mean',
 'chroma_stft_var',
 'rms_mean',
 'rms_var',
 'spectral_centroid_mean',
 'spectral_centroid_var',
 'spectral_bandwidth_mean',
 'spectral_bandwidth_var',
 'rolloff_mean',
 'rolloff_var',
 'zero_crossing_rate_mean',
 'zero_crossing_rate_var',
 'spectral_contrast_mean',
 'spectral_contrast_var',
 'spectral_flatness_mean',
 'spectral_flatness_var',
 'tempo',
 'mfcc1_mean',
 'mfcc1_var',
 'mfcc2_mean',
 'mfcc2_var',
 'mfcc3_mean',
 'mfcc3_var',
 'mfcc4_mean',
 'mfcc4_var',
 'mfcc5_mean',
 'mfcc5_var',
 'mfcc6_mean',
 'mfcc6_var',
 'mfcc7_mean',
 'mfcc7_var',
 'mfcc8_mean',
 'mfcc8_var',
 'mfcc9_mean',
 'mfcc9_var',
 'mfcc10_mean',
 'mfcc10_var',
 'mfcc11_mean',
 'mfcc11_var',
 'mfcc12_mean',
 'mfcc12_var',
 'mfcc13_mean',
 'mfcc13_var',
 'mfcc14_mean',
 'mfcc14_var',
 'mfcc15_mean',
 'mfcc15_var',
 'mfcc16_mean',
 'mfcc16_var',
 'mfcc17_mean',
 'mfcc17_var',
 'mfcc18_mean',
 'mfcc18_var',
 'mfcc19_mean',
 'mfcc19_var',
 'mfcc20_mean',
 'mfcc20_var',
 'label

In [105]:
genres = {0: 'blues',
 1: 'classical',
 2: 'country',
 3: 'disco',
 4: 'hiphop',
 5: 'jazz',
 6: 'metal',
 7: 'pop',
 8: 'reggae',
 9: 'rock'}

In [106]:
genres.values()

dict_values(['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'])

In [107]:
directory = dict()
all_files = []
for g in genres.values():
    files = os.listdir(f'gtzan_300ms_150ms_overlap/data/genres/{g}')
    directory[g] = files
    for f in files:
        all_files.append(f"{g}/{f}")

all_files.sort()
len(directory['pop'])

20000

In [108]:
all_files[0:20]

['blues/blues.00000.wav',
 'blues/blues.00001.wav',
 'blues/blues.00002.wav',
 'blues/blues.00003.wav',
 'blues/blues.00004.wav',
 'blues/blues.00005.wav',
 'blues/blues.00006.wav',
 'blues/blues.00007.wav',
 'blues/blues.00008.wav',
 'blues/blues.00009.wav',
 'blues/blues.00010.wav',
 'blues/blues.00011.wav',
 'blues/blues.00012.wav',
 'blues/blues.00013.wav',
 'blues/blues.00014.wav',
 'blues/blues.00015.wav',
 'blues/blues.00016.wav',
 'blues/blues.00017.wav',
 'blues/blues.00018.wav',
 'blues/blues.00019.wav']

In [121]:
def feature_extraction(df, files):
    main_folder = f'gtzan_300ms_150ms_overlap/data/genres/'
    n_fft = 256
    hop_length = 128
    
    for file in files:
        new_features = dict()
        #file_path = gtzan_300ms_150ms_overlap/data/genres/{genre}/{filename}
        file_path = main_folder+file

        try:
            ly, lsr = librosa.load(file_path)
        
            label, filename = file.split("/")
            
            new_features['filename'] = filename
        
            #Chroma_STFT
            try:
                chroma_stft = librosa.feature.chroma_stft(y=ly, sr=lsr, n_fft=n_fft, hop_length=hop_length)

                new_features['chroma_stft_mean'] = np.mean(chroma_stft)
                new_features['chroma_stft_var'] = np.var(chroma_stft)
    
            except Exception as e:
                print(f"Failed to process Chroma STFT for {file_path}: {e}")
        
            #RMS
            try:
                rootms = librosa.feature.rms(y=ly, hop_length=hop_length)
    
                new_features['rms_mean'] = np.mean(rootms)
                new_features['rms_var'] = np.var(rootms)
    
            except Exception as e:
                print(f"Failed to process Root Mean Square for {file_path}: {e}")
            
            #Spectral_Centroid
            try:
                spec_cen = librosa.feature.spectral_centroid(y=ly, sr=lsr, n_fft=n_fft, hop_length=hop_length)
    
                new_features['spectral_centroid_mean'] = np.mean(spec_cen)
                new_features['spectral_centroid_var'] = np.var(spec_cen)
    
            except Exception as e:
                print(f"Failed to process Spectral Centroid for {file_path}: {e}")
            
            #Spectral_Bandwidth
            try:
                spec_bw = librosa.feature.spectral_bandwidth(y=ly, sr=lsr, n_fft=n_fft, hop_length=hop_length)
    
                new_features['spectral_bandwidth_mean'] = np.mean(spec_bw)
                new_features['spectral_bandwidth_var'] = np.var(spec_bw)
    
            except Exception as e:
                print(f"Failed to process Spectral Bandwidth for {file_path}: {e}")
                
            #Rolloff
            try:
                rolloff = librosa.feature.spectral_rolloff(y=ly, sr=lsr, n_fft=n_fft, hop_length=hop_length)
    
                new_features['rolloff_mean'] = np.mean(rolloff)
                new_features['rolloff_var'] = np.var(rolloff)
    
            except Exception as e:
                print(f"Failed to process Spectral Rolloff for {file_path}: {e}")
            
            #Zero_Crossing_Rate
            try:
                zcr = librosa.feature.zero_crossing_rate(y=ly, hop_length=hop_length)
             
                new_features['zero_crossing_rate_mean'] = np.mean(zcr)
                new_features['zero_crossing_rate_var'] = np.var(zcr)
    
            except Exception as e:
                print(f"Failed to process Zero Crossing Rate for {file_path}: {e}")

            #Spectral Flatness
            try:
                spec_fl = librosa.feature.spectral_flatness(y=ly, n_fft=n_fft, hop_length=hop_length)
    
                new_features['spectral_flatness_mean'] = np.mean(spec_fl)
                new_features['spectral_flatness_var'] = np.var(spec_fl)
    
            except Exception as e:
                print(f"Failed to process Spectral Flatness for {file_path}: {e}")
            
            #Spectral Contrast
            try:
                spec_co = librosa.feature.spectral_contrast(y=ly, sr=lsr, n_fft=n_fft, hop_length=hop_length)
    
                new_features['spectral_contrast_mean'] = np.mean(spec_co)
                new_features['spectral_contrast_var'] = np.var(spec_co)
    
            except Exception as e:
                print(f"Failed to process Spectral Contrast for {file_path}: {e}")

            #Tempo (No Mean/Variance)
            try:
                onset_env = librosa.onset.onset_strength(y=ly, sr=lsr)
                tem = librosa.feature.tempo(onset_envelope=onset_env, sr=lsr, hop_length=hop_length)
    
                new_features['tempo'] = tem
    
            except Exception as e:
                print(f"Failed to process Tempo for {file_path}: {e}")

            #MFCCs (20 of them)
            try:
                mfccs = librosa.feature.mfcc(y=ly, sr=lsr, n_fft=n_fft, hop_length=hop_length)

                for i, mfcc in enumerate(mfccs):
                    new_features[f'mfcc{i+1}_mean'] = np.mean(mfcc)
                    new_features[f'mfcc{i+1}_var'] = np.var(mfcc)
    
            except Exception as e:
                print(f"Failed to process MFCCs for {file_path}: {e}")
            
            #Label
            new_features['label'] = label
            
            new_row = pd.DataFrame([new_features])
            df = pd.concat([df, new_row], ignore_index=True)

        except Exception as e:
            print(f"Error loading audio file({file_path}) in librosa: {e}")

    return df


In [122]:
features = features.iloc[:0]
features = feature_extraction(features, all_files)

/opt/anaconda3/envs/csc_7760_final/lib/python3.12/site-packages/librosa/feature/spectral.py:2148: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)
/opt/anaconda3/envs/csc_7760_final/lib/python3.12/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=176
  warnings.warn(
/opt/anaconda3/envs/csc_7760_final/lib/python3.12/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=860
  warnings.warn(
/opt/anaconda3/envs/csc_7760_final/lib/python3.12/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=2028
  warnings.warn(
/opt/anaconda3/envs/csc_7760_final/lib/python3.12/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input

In [124]:
features

,filename,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,rolloff_mean,...,mfcc16_var,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var,label
0,blues.00000.wav,0.820354,0.022998,0.141991,0.010220,1773.838925,438338.716796,2367.591548,77382.577395,4359.645433,...,34.716759,-8.558198,36.560204,0.433451,24.618813,-5.878705,45.574993,-10.892500,47.030720,blues
1,blues.00001.wav,0.716320,0.044223,0.147373,0.009406,2148.145299,252431.482835,2263.753810,135025.397437,4705.833083,...,42.564156,-12.492635,83.172112,-2.281379,34.923225,-5.373457,36.997662,-8.973206,52.851116,blues
2,blues.00002.wav,0.624650,0.050467,0.191377,0.003466,2049.022054,104157.843550,1923.581746,8168.492742,3862.725361,...,70.133766,-15.582106,72.812714,-1.749841,43.884888,-5.319300,30.090799,-10.574518,50.951218,blues
3,blues.00003.wav,0.682139,0.044001,0.162216,0.001825,2229.533424,100136.714803,2090.903716,59013.059900,4409.337440,...,68.789055,-14.845062,34.785408,3.200582,38.081703,-6.502884,34.439751,-10.010311,44.665058,blues
4,blues.00004.wav,0.547533,0.097336,0.172501,0.000864,2434.913717,109626.588906,2090.908506,75809.998182,4510.377855,...,71.420647,-10.703167,43.099609,5.970252,47.964809,-7.772474,41.583679,-7.409538,40.371536,blues
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200572,rock.20098.wav,0.734123,0.028219,0.190071,0.002103,1727.590742,345610.214188,1927.301900,170924.061128,3327.708083,...,35.670769,-0.602435,45.425331,3.055514,36.634285,-4.690790,39.660339,-7.668109,41.779305,rock
200573,rock.20099.wav,0.670236,0.038048,0.207446,0.000954,2078.777558,232584.591133,2046.169252,127235.288551,3958.796575,...,33.821934,-8.180123,89.212173,2.103449,32.583015,-0.618241,65.927376,-6.041405,42.973103,rock
200574,rock.20100.wav,0.581977,0.061297,0.199906,0.001040,2102.580105,35622.752726,1919.052664,62320.620404,3872.663762,...,20.396629,-14.413528,32.151768,0.552173,40.916851,3.789587,36.593746,-4.513898,29.671715,rock
200575,rock.20101.wav,0.546039,0.068201,0.195705,0.001995,2174.693698,49751.526481,2001.873376,75464.591633,4048.242188,...,29.443493,-10.898533,53.397324,-3.101555,29.256279,-0.597299,34.534416,-4.171192,35.655727,rock


In [145]:
idx = features.columns.get_loc('filename') + 1

features.insert(idx, 'length', 300)

In [146]:
features

,filename,length,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,...,mfcc16_var,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var,label
0,blues.00000.wav,300,0.820354,0.022998,0.141991,0.010220,1773.838925,438338.716796,2367.591548,77382.577395,...,34.716759,-8.558198,36.560204,0.433451,24.618813,-5.878705,45.574993,-10.892500,47.030720,blues
1,blues.00001.wav,300,0.716320,0.044223,0.147373,0.009406,2148.145299,252431.482835,2263.753810,135025.397437,...,42.564156,-12.492635,83.172112,-2.281379,34.923225,-5.373457,36.997662,-8.973206,52.851116,blues
2,blues.00002.wav,300,0.624650,0.050467,0.191377,0.003466,2049.022054,104157.843550,1923.581746,8168.492742,...,70.133766,-15.582106,72.812714,-1.749841,43.884888,-5.319300,30.090799,-10.574518,50.951218,blues
3,blues.00003.wav,300,0.682139,0.044001,0.162216,0.001825,2229.533424,100136.714803,2090.903716,59013.059900,...,68.789055,-14.845062,34.785408,3.200582,38.081703,-6.502884,34.439751,-10.010311,44.665058,blues
4,blues.00004.wav,300,0.547533,0.097336,0.172501,0.000864,2434.913717,109626.588906,2090.908506,75809.998182,...,71.420647,-10.703167,43.099609,5.970252,47.964809,-7.772474,41.583679,-7.409538,40.371536,blues
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200572,rock.20098.wav,300,0.734123,0.028219,0.190071,0.002103,1727.590742,345610.214188,1927.301900,170924.061128,...,35.670769,-0.602435,45.425331,3.055514,36.634285,-4.690790,39.660339,-7.668109,41.779305,rock
200573,rock.20099.wav,300,0.670236,0.038048,0.207446,0.000954,2078.777558,232584.591133,2046.169252,127235.288551,...,33.821934,-8.180123,89.212173,2.103449,32.583015,-0.618241,65.927376,-6.041405,42.973103,rock
200574,rock.20100.wav,300,0.581977,0.061297,0.199906,0.001040,2102.580105,35622.752726,1919.052664,62320.620404,...,20.396629,-14.413528,32.151768,0.552173,40.916851,3.789587,36.593746,-4.513898,29.671715,rock
200575,rock.20101.wav,300,0.546039,0.068201,0.195705,0.001995,2174.693698,49751.526481,2001.873376,75464.591633,...,29.443493,-10.898533,53.397324,-3.101555,29.256279,-0.597299,34.534416,-4.171192,35.655727,rock


In [147]:
features.to_csv('features_300ms_150ms_overlap.csv', index=False)